# Estudo de Caso 2.1 — Classificador de Fluidos Não-Newtonianos

**Capítulo Associado:** Capítulo 2 — Fundamentos dos Fluidos e Viscosidade  
**Nível:** Graduação  
**Tempo estimado:** 30 minutos  
**Autor:** Jader Lugon Junior  

---

## 📋 Resumo

Neste estudo de caso, você irá:

1. Receber dados experimentais fictícios de tensão de cisalhamento vs. taxa de deformação
2. Ajustar os dados ao modelo de Lei de Potência (Ostwald-de Waele)
3. Classificar o fluido como pseudoplástico, dilatante ou newtoniano
4. Visualizar o ajuste e os parâmetros reológicos

---

## 🎯 Objetivos de Aprendizagem

- Aplicar regressão não-linear para ajuste de modelos reológicos
- Interpretar o índice de comportamento do fluxo ($n$)
- Identificar singularidades matemáticas em modelos reológicos
- Discutir limitações de modelos empíricos

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `scipy`, `matplotlib`, `pandas`
- Conhecimento prévio: Lei da Viscosidade de Newton (Cap. 2)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Ambiente configurado com sucesso!")

## 📊 Dados Experimentais

Considere os seguintes dados de um fluido desconhecido:

In [ ]:
# ============================================================
# DADOS EXPERIMENTAIS
# ============================================================

# Dados experimentais (fictícios)
gamma_dot = np.array([0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0, 500.0])  # s⁻¹
tau = np.array([0.8, 3.5, 6.2, 22.5, 38.0, 120.0, 200.0, 650.0])      # Pa

df = pd.DataFrame({
    'Taxa de Deformação [s⁻¹]': gamma_dot,
    'Tensão de Cisalhamento [Pa]': tau
})

print("📊 Dados experimentais:")
display(df)

## 🧮 Modelo de Lei de Potência (Ostwald-de Waele)

O modelo é dado por:

$$\tau = K \cdot \dot{\gamma}^n$$

Onde:
- $K$ = índice de consistência [Pa·sⁿ]
- $n$ = índice de comportamento do fluxo [adimensional]

**Classificação:**
- $n < 1$: Pseudoplástico (cisalhamento-fino)
- $n = 1$: Newtoniano
- $n > 1$: Dilatante (cisalhamento-espesso)

In [ ]:
# ============================================================
# AJUSTE DO MODELO
# ============================================================

def lei_potencia(gamma, K, n):
    """Modelo de Lei de Potência (Ostwald-de Waele)"""
    return K * gamma**n

# Ajuste não-linear
popt, pcov = curve_fit(lei_potencia, gamma_dot, tau, p0=[1.0, 1.0])
K_fit, n_fit = popt

print(f"📊 Parâmetros ajustados:")
print(f"  • K = {K_fit:.4f} Pa·sⁿ")
print(f"  • n = {n_fit:.4f}")

# Classificação
if n_fit < 0.95:
    classificacao = "PSEUDOPLÁSTICO (cisalhamento-fino)"
elif n_fit > 1.05:
    classificacao = "DILATANTE (cisalhamento-espesso)"
else:
    classificacao = "NEWTONIANO"

print(f"\n🏷️ Classificação: {classificacao}")

## 📈 Visualização do Ajuste

In [ ]:
# ============================================================
# VISUALIZAÇÃO
# ============================================================

# Gerar curva de ajuste
gamma_fit = np.logspace(-1, 3, 100)
tau_fit = lei_potencia(gamma_fit, K_fit, n_fit)

fig, ax = plt.subplots(figsize=(10, 6))

# Dados experimentais
ax.scatter(gamma_dot, tau, s=100, c='red', marker='o', 
           label='Dados experimentais', zorder=5)

# Curva de ajuste
ax.plot(gamma_fit, tau_fit, 'b-', linewidth=2, 
        label=f'Ajuste: $\\tau = {K_fit:.2f} \\cdot \\dot{{\\gamma}}^{{{n_fit:.2f}}}$')

ax.set_xlabel('Taxa de Deformação, $\\dot{\\gamma}$ [s⁻¹]', fontsize=12)
ax.set_ylabel('Tensão de Cisalhamento, $\\tau$ [Pa]', fontsize=12)
ax.set_title('Ajuste do Modelo de Lei de Potência', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ⚠️ Armadilha: Singularidade em $\dot{\gamma} = 0$

Um erro comum é calcular a viscosidade aparente em $\dot{\gamma} = 0$:

$$\mu_{ap} = \frac{\tau}{\dot{\gamma}} = K \cdot \dot{\gamma}^{n-1}$$

Para $n < 1$ (pseudoplástico), quando $\dot{\gamma} \to 0$, $\mu_{ap} \to \infty$.

**Solução:** Usar modelos mais robustos como Carreau ou Cross, que limitam a viscosidade:

$$\mu(\dot{\gamma}) = \mu_\infty + \frac{\mu_0 - \mu_\infty}{[1 + (\lambda \dot{\gamma})^2]^{(n-1)/2}}$$

In [ ]:
# ============================================================
# DEMONSTRAÇÃO DA SINGULARIDADE
# ============================================================

print("=" * 60)
print("DEMONSTRAÇÃO DA SINGULARIDADE")
print("=" * 60)

# Viscosidade aparente em diferentes taxas
gamma_test = np.array([0.001, 0.01, 0.1, 1.0, 10.0])
mu_ap = K_fit * gamma_test**(n_fit - 1)

print("\n📊 Viscosidade aparente em diferentes taxas de deformação:")
print(f"{'γ̇ [s⁻¹]':<15} {'μ_ap [Pa·s]':<15}")
print("-" * 30)
for g, mu in zip(gamma_test, mu_ap):
    print(f"{g:15.3f} {mu:15.2f}")

print(f"\n💡 Observe que para γ̇ → 0, μ_ap → ∞ (singularidade)")
print(f"   Isso é uma limitação do modelo de Lei de Potência!")

## 💡 Discussão

### Limitações do Modelo de Lei de Potência

1. **Não descreve comportamento em baixas taxas**: A viscosidade tende ao infinito
2. **Não captura tensão de escoamento**: Fluidos como ketchup têm $\tau_0 > 0$
3. **Válido apenas em faixa intermediária**: Em $\dot{\gamma}$ muito altos, $\mu \to \mu_\infty$

### Alternativas

- **Herschel-Bulkley**: $\tau = \tau_0 + K \dot{\gamma}^n$ (com tensão de escoamento)
- **Carreau-Yasuda**: Descreve platôs em baixas e altas taxas
- **Bingham**: Para plásticos ideais ($\tau = \tau_0 + \mu_p \dot{\gamma}$)

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 2](../notebooks/02_Fundamentos_Fluidos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 2.1**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>

In [ ]:
print("=" * 60)
print("✅ ESTUDO DE CASO CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Estudo de Caso 2.1!")
print("\n💡 Próximo passo: Explore outros estudos de caso do Capítulo 2")